# Diniz Ch.6 — AP, SM-AP and partial-update SM-AP

比較 χ=1、20、80，保留 projection orders、membership bounds 與每次最多 5 個係數更新。

來源：`Diniz-Chap6-SysID.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Parameters

In [ ]:
from adsp.experiments import diniz

n_samples, n_trials, seed, noise_var = 3000, 20, 6036, 1e-4
# Original notebook used 100 trials; set n_trials=100 for that ensemble size.
spread_cases = (1, 20, 80)
grouped_results = {}

## 3. Unified Monte Carlo for all projection variants

In [ ]:
for spread in spread_cases:
  make_data = partial(data.system_identification, n_samples, h=diniz.SYSTEM_TAPS, pole=diniz.pole_for_spread(spread),
                      noise_var=noise_var, burn_in=512)
  models = diniz.membership_models(noise_var=noise_var, spread=spread)
  grouped_results[spread] = training.compare(models, make_data, n_trials=n_trials, seed=seed)

## 4. MSE, update rate and selected coefficients

In [ ]:
for spread, results in grouped_results.items():
  fig, grid = plot.axes(rows=2)
  plot.learning(results, ax=grid[0, 0], noise_floor=noise_var, window=64, title=f"Membership filters: χ={spread}")
  updates = {name: result.extras["update_flags"] for name, result in results.items() if "update_flags" in result.extras}
  if updates:
    plot.lines(updates, ax=grid[1, 0], window=64, ylabel="Update probability", title="Membership update rate")
  plot.show(fig)
  training.summary(results, noise_floor=noise_var)
  for name, result in results.items():
    if "selected_count" in result.extras:
      print(name, "mean selected coefficient count:", np.mean(result.extras["selected_count"]))